In [2]:
%pip install ollama
%pip install pydantic

  Using cached h11-0.14.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_extensions-4.12.2-py3-none-any.whl.metadata (3.0 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.3 MB/s eta 0:00:00
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_extensions-4.12.2-py3-none-any.whl (37 kB)
Using cached h11-0.14.0-py3-none-any.whl (58 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)

[notice] A new release of pip is available: 24.2 -> 25.0
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 25.0
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from typing import Optional
from ollama import chat # type: ignore
from pydantic import BaseModel # type: ignore

## Example Playing Around

In [4]:
class Pet(BaseModel):
    name: str
    specie: str
    age: int
    fur_color: Optional[str] = None 
    favorite_toy: Optional[str] = None 

class PetList(BaseModel):
    pets: list[Pet]

In [5]:
def pet_response(content):
    response = chat(
        messages=[
            {
                'role': 'user',
                'content': content,
            }
        ],
        model = 'llama3.2',
        format = PetList.model_json_schema(),
    )
    pets = PetList.model_validate_json(response.message.content)
    return pets

## Test Pet Detection

In [6]:
pet_response("I have a black cat named Tom who is only 5 months old. For his birthday, I want to give him his favorite toy, which is a ball of yarn.")

PetList(pets=[Pet(name='Tom', specie='cat', age=5, fur_color=None, favorite_toy=None), Pet(name='yarn ball', specie='toy', age=0, fur_color=None, favorite_toy=None)])

## Marker Gene Extraction

In [7]:
class MarkerGene(BaseModel):
    marker_gene_name: Optional[str] = None
    cell_name: Optional[str] = None
    tissue_source: Optional[str] = None
    # try to add annotations/descriptors to help LLM better identify it

class MarkerGeneList(BaseModel):
    genes: list[MarkerGene]

In [8]:
def gene_response(paper_content, prompt):
    response = chat(
        messages=[
            {
                'role': 'system',
                'content': prompt,
            },
            {
                'role': 'user',
                'content': paper_content,
            }
        ],
        model = 'llama3.2',
        format = MarkerGeneList.model_json_schema()
    )
    genes = MarkerGeneList.model_validate_json(response.message.content)
    return genes

In [10]:
gene_response('''After doublet removal and quality filtering, we considered a total of 197,721 cells (106,469 from PG and 91,252 from ING), identifying all cell types observed in human WAT (Fig. 2c, d, Supplementary Table 2) with the addition of distinct male and female epithelial populations (Dcdc2a+ and Erbb4+, respectively)''', "You are a genomics researcher trying to discover different cell types and what genes they express, aka find the marker genes within the given sentence.")

MarkerGeneList(genes=[MarkerGene(marker_gene_name='Dcdc2a', cell_name='male epithelial population', tissue_source=None), MarkerGene(marker_gene_name='Erbb4+', cell_name='female epithelial population', tissue_source=None)])